# MedCompress — BraTS 2021 Brain Tumor Segmentation
**Author: Abhishek Shekhar**

2.5D U-Net on BraTS 2021 MRI (T1/T1ce/T2/FLAIR). 4-class segmentation.

> **Note:** This is the most compute-intensive notebook (~6-8h on T4). Use Colab Pro+ or run in multiple sessions — checkpoints persist on Drive.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "brats_segmentation"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/brats_segmentation"
DATA_DIR    = f"/content/data/brats_segmentation"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization nibabel SimpleITK tf2onnx onnx tqdm
import tensorflow_model_optimization as tfmot
import nibabel as nib
print("Packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Download BraTS 2021 ─────────────────────────────────────────────────────
BRATS_DIR = f"{DATA_DIR}/brats"
os.makedirs(BRATS_DIR, exist_ok=True)
if not any(os.scandir(BRATS_DIR)):
    print("Downloading BraTS 2021...")
    # Try multiple dataset slugs
    for slug in ["dschettler8845/brats-2021-task1","awsaf49/brats2020-training-data",
                 "andrewmvd/brain-tumor-segmentation-in-mri-brats-2015"]:
        try:
            !kaggle datasets download -d {slug} -p {BRATS_DIR} && break
        except: pass
    !cd {BRATS_DIR} && unzip -q "*.zip" && rm -f *.zip
    print("Download complete ✓")
else:
    print(f"BraTS cached ✓")

# Find NIfTI files
import glob
t1_files = glob.glob(f"{BRATS_DIR}/**/*_t1.nii*",recursive=True)[:10]
print(f"Found {len(t1_files)} NIfTI samples (showing first 10)")
for f in t1_files[:3]: print(f"  {f}")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
import nibabel as nib
PATCH   = 128
N_SLICES= 3
MODS    = ["t1","t1ce","t2","flair"]
N_CH    = N_SLICES*len(MODS)   # 12 input channels
N_CLS   = 4                     # BG, NCR, ED, ET
BATCH   = 8
SEEDS   = [42, 123, 456]        # 3 seeds (BraTS is compute-heavy)
LABEL_REMAP = {0:0,1:1,2:2,4:3}

CFG = {
    "baseline": {"epochs":50,"lr":1e-4,"patience":10},
    "qat":      {"epochs":10,"lr":1e-5},
    "student":  {"epochs":50,"lr":1e-4,"patience":10},
    "kd":       {"epochs":50,"lr":1e-4,"patience":10,"temperature":3.0,"alpha":0.6},
}
CLASS_WEIGHTS = np.array([0.1,3.0,1.5,3.0],dtype=np.float32)

def set_seed(s): random.seed(s); np.random.seed(s); tf.random.set_seed(s)

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── BraTS data loading ───────────────────────────────────────────────────────
import glob, re
from sklearn.model_selection import train_test_split

def find_patient_dirs(root):
    """Find directories containing all 4 modalities + seg."""
    patients=[]
    for d,_,files in os.walk(root):
        has = {m: any(m in f for f in files) for m in MODS}
        has_seg = any("seg" in f for f in files)
        if all(has.values()) and has_seg: patients.append(d)
        if len(patients)>5: break  # limit scan depth speed
    if not patients:
        # Fallback: group by patient ID
        all_t1 = sorted(glob.glob(f"{BRATS_DIR}/**/*t1.nii*",recursive=True))
        for t1 in all_t1:
            d=os.path.dirname(t1)
            if d not in patients: patients.append(d)
    return patients

patients = find_patient_dirs(BRATS_DIR)
print(f"Found {len(patients)} patients")
if len(patients)<5: print("⚠ Low patient count — check download and dataset structure")

train_pts,test_pts=train_test_split(patients,test_size=0.15,random_state=42)
train_pts,val_pts =train_test_split(train_pts,test_size=0.15/0.85,random_state=42)
print(f"Splits: train={len(train_pts)} val={len(val_pts)} test={len(test_pts)}")

In [ ]:
# ── Slice extractor → tf.data ────────────────────────────────────────────────
def load_patient_slices(patient_dir):
    """Load 2.5D slices from one patient. Returns (N,128,128,12) imgs, (N,128,128,4) one-hot."""
    pd_str = patient_dir.decode() if isinstance(patient_dir, bytes) else patient_dir
    def find_file(mod):
        for f in os.listdir(pd_str):
            if mod in f and f.endswith(('.nii','.nii.gz')): return os.path.join(pd_str,f)
        return None
    vol_data = []
    for m in MODS:
        fp=find_file(m)
        if fp is None: return None,None
        v=nib.load(fp).get_fdata(dtype=np.float32)
        p1,p99=np.percentile(v[v>0],[1,99]) if v.max()>0 else (0,1)
        v=np.clip(v,p1,p99); v=(v-v.mean())/(v.std()+1e-8)
        vol_data.append(v)
    seg_fp=find_file("seg")
    if seg_fp is None: return None,None
    seg=nib.load(seg_fp).get_fdata().astype(np.int32)
    seg_remapped=np.zeros_like(seg)
    for old,new in LABEL_REMAP.items(): seg_remapped[seg==old]=new
    H,W,D=vol_data[0].shape
    imgs,masks=[],[]
    for z in range(N_SLICES//2, D-N_SLICES//2, 2):  # stride 2 to reduce volume
        slices=[]
        for m_data in vol_data:
            for dz in range(-N_SLICES//2, N_SLICES//2+1):
                slices.append(m_data[:,:,z+dz])
        patch=np.stack(slices,-1)[:PATCH,:PATCH,:]  # crop/pad
        if patch.shape[0]<PATCH or patch.shape[1]<PATCH:
            pad=np.zeros((PATCH,PATCH,N_CH),np.float32)
            pad[:patch.shape[0],:patch.shape[1],:]=patch; patch=pad
        s=seg_remapped[:PATCH,:PATCH,z]
        if s.shape[0]<PATCH or s.shape[1]<PATCH:
            sp=np.zeros((PATCH,PATCH),np.int32); sp[:s.shape[0],:s.shape[1]]=s; s=sp
        one_hot=np.eye(N_CLS,dtype=np.float32)[s]
        imgs.append(patch); masks.append(one_hot)
    return np.array(imgs,np.float32), np.array(masks,np.float32)

def make_ds(patient_list, shuffle=False):
    all_imgs,all_masks=[],[]
    for p in patient_list:
        imgs,masks=load_patient_slices(p)
        if imgs is not None: all_imgs.append(imgs); all_masks.append(masks)
    if not all_imgs: raise ValueError("No data loaded!")
    X=np.concatenate(all_imgs); Y=np.concatenate(all_masks)
    print(f"  Loaded {len(X)} slices from {len(all_imgs)} patients")
    ds=tf.data.Dataset.from_tensor_slices((X,Y))
    if shuffle: ds=ds.shuffle(len(X))
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

print("Loading datasets (may take a few minutes)...")
train_ds=make_ds(train_pts,True); val_ds=make_ds(val_pts); test_ds=make_ds(test_pts)

In [ ]:
# ── 2.5D U-Net ───────────────────────────────────────────────────────────────
from tensorflow.keras import layers

def conv2d_bn(x,f,k=3):
    x=layers.Conv2D(f,k,padding="same",use_bias=False)(x)
    return layers.Activation("relu")(layers.BatchNormalization()(x))

def build_unet25d(filters=(32,64,128,256), name="unet25d"):
    inp=keras.Input((PATCH,PATCH,N_CH))
    skips,x=[],inp
    for f in filters:
        x=conv2d_bn(conv2d_bn(x,f),f); skips.append(x); x=layers.MaxPooling2D(2)(x)
    x=conv2d_bn(conv2d_bn(x,filters[-1]*2),filters[-1]*2)
    for f in reversed(filters):
        x=layers.Conv2DTranspose(f,2,2,padding="same")(x)
        x=layers.Concatenate()([x,skips.pop()])
        x=conv2d_bn(conv2d_bn(x,f),f)
    out=layers.Conv2D(N_CLS,1,activation="softmax")(x)
    return keras.Model(inp,out,name=name)

def build_unet25d_lite(name="unet25d_lite"):
    return build_unet25d((16,32,64,128),name=name)

def wce_dice_loss(y_true,y_pred):
    weights=tf.constant(CLASS_WEIGHTS)
    wce=tf.reduce_mean(-tf.reduce_sum(weights*y_true*tf.math.log(y_pred+1e-7),axis=-1))
    inter=tf.reduce_sum(y_true*y_pred,axis=(0,1,2))
    dice=1-tf.reduce_mean((2*inter+1e-7)/(tf.reduce_sum(y_true+y_pred,axis=(0,1,2))+1e-7))
    return wce+dice

def eval_seg(model, ds, tag=""):
    dices=[]
    for imgs,masks in ds:
        p=model(imgs,training=False).numpy(); m=masks.numpy()
        pb=(p==p.max(-1,keepdims=True)).astype(np.float32)
        for c in range(1,N_CLS):  # skip background
            inter=(pb[...,c]*m[...,c]).sum(); union=(pb[...,c]+m[...,c]).sum()
            dices.append((2*inter+1e-7)/(union+1e-7))
    mean_d=np.mean(dices)
    print(f"  [{tag}] Mean Dice (fg)={mean_d:.4f}")
    return {"mean_dice":float(mean_d)}

print("2.5D U-Net defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENTS — BraTS (Baseline → Student Scratch → KD → QAT)
# ══════════════════════════════════════════════════════════════════════════════
import tensorflow_model_optimization as tfmot
baseline_r, scratch_r, kd_r, qat_r = [], [], [], []

for seed in SEEDS:
    set_seed(seed)
    # ─ Baseline ─
    key=f"baseline_s{seed}"
    if is_done(key): baseline_r.append(load_result(key)); print(f"⏩ {key}")
    else:
        m=build_unet25d()
        m.compile(keras.optimizers.Adam(CFG["baseline"]["lr"]),wce_dice_loss)
        m,_=train_model(m,train_ds,val_ds,CFG["baseline"],key,"val_loss","min")
        metrics=eval_seg(m,test_ds,key); m.save(f"{MODELS_DIR}/unet25d_s{seed}.keras")
        mark_done(key,metrics); baseline_r.append(metrics)
    # ─ Student Scratch ─
    key=f"scratch_s{seed}"
    if is_done(key): scratch_r.append(load_result(key)); print(f"⏩ {key}")
    else:
        s=build_unet25d_lite()
        s.compile(keras.optimizers.Adam(CFG["student"]["lr"]),wce_dice_loss)
        s,_=train_model(s,train_ds,val_ds,CFG["student"],key,"val_loss","min")
        metrics=eval_seg(s,test_ds,key); s.save(f"{MODELS_DIR}/unet25d_lite_scratch_s{seed}.keras")
        mark_done(key,metrics); scratch_r.append(metrics)

print(f"Baseline: {np.mean([r['mean_dice'] for r in baseline_r]):.4f}")
print(f"Scratch:  {np.mean([r['mean_dice'] for r in scratch_r]):.4f}")

In [ ]:
# ─ KD + QAT for BraTS ───────────────────────────────────────────────────────
teacher=keras.models.load_model(f"{MODELS_DIR}/unet25d_s42.keras",
                                 custom_objects={"wce_dice_loss":wce_dice_loss})
T=CFG["kd"]["temperature"]; ALPHA=CFG["kd"]["alpha"]

for seed in SEEDS:
    key=f"kd_s{seed}"
    if is_done(key): kd_r.append(load_result(key)); print(f"⏩ {key}"); continue
    set_seed(seed)
    student=build_unet25d_lite(); opt=keras.optimizers.Adam(CFG["kd"]["lr"])
    best_loss,patience_c=1e9,0; ckpt=f"{CKPT_DIR}/kd_s{seed}.keras"
    if os.path.exists(ckpt): student.load_weights(ckpt); print("  Resumed ✓")
    for epoch in range(CFG["kd"]["epochs"]):
        for imgs,masks in train_ds:
            with tf.GradientTape() as tape:
                t_out=teacher(imgs,training=False); s_out=student(imgs,training=True)
                ts=tf.nn.softmax(tf.math.log(t_out+1e-7)/T)
                ss=tf.nn.softmax(tf.math.log(s_out+1e-7)/T)
                kl=tf.reduce_mean(tf.reduce_sum(ts*tf.math.log((ts+1e-7)/(ss+1e-7)),-1))
                loss=ALPHA*kl*(T**2)+(1-ALPHA)*wce_dice_loss(masks,s_out)
            opt.apply_gradients(zip(tape.gradient(loss,student.trainable_variables),student.trainable_variables))
        vl=np.mean([wce_dice_loss(m,student(i,training=False)).numpy() for i,m in val_ds])
        if vl<best_loss: best_loss=vl; patience_c=0; student.save_weights(ckpt)
        else:
            patience_c+=1
            if patience_c>=CFG["kd"]["patience"]: break
    student.load_weights(ckpt); metrics=eval_seg(student,test_ds,key)
    student.save(f"{MODELS_DIR}/unet25d_lite_kd_s{seed}.keras"); mark_done(key,metrics); kd_r.append(metrics)

for seed in SEEDS:
    key=f"qat_kd_s{seed}"
    if is_done(key): qat_r.append(load_result(key)); print(f"⏩ {key}"); continue
    base=keras.models.load_model(f"{MODELS_DIR}/unet25d_lite_kd_s{seed}.keras",
                                  custom_objects={"wce_dice_loss":wce_dice_loss})
    qat_m=tfmot.quantization.keras.quantize_model(base)
    qat_m.compile(keras.optimizers.Adam(1e-5),wce_dice_loss)
    qat_m.fit(train_ds,validation_data=val_ds,epochs=10,verbose=1)
    stripped=tfmot.quantization.keras.strip_pruning(qat_m)
    p=f"{TFLITE_DIR}/unet25d_lite_kd_int8_s{seed}.tflite"
    conv=tf.lite.TFLiteConverter.from_keras_model(stripped)
    conv.optimizations=[tf.lite.Optimize.DEFAULT]
    with open(p,"wb") as f: f.write(conv.convert())
    metrics=eval_seg(stripped,test_ds,key); metrics["size_mb"]=os.path.getsize(p)/1e6
    mark_done(key,metrics); qat_r.append(metrics)

table=pd.DataFrame([{"method":"Baseline","dice":np.mean([r["mean_dice"] for r in baseline_r])},
                     {"method":"Scratch","dice":np.mean([r["mean_dice"] for r in scratch_r])},
                     {"method":"KD","dice":np.mean([r["mean_dice"] for r in kd_r])},
                     {"method":"KD+QAT","dice":np.mean([r.get("mean_dice",0) for r in qat_r])}])
table.to_csv(f"{RESULTS_DIR}/brats_results.csv",index=False)
print(table.to_string(index=False)); print(f"Saved: {RESULTS_DIR}")